# Modulo 2 — Feature Engineering

Estrazione automatica di feature dai dati OHLCV del Modulo 1: indicatori tecnici, pattern candlestick e **market structure** (swing, BOS/CHoCH, FVG, S/R, liquidità, volatilità, trend strength).

Le feature sono registrate in un **registry estensibile**: aggiungerne una nuova = un decoratore `@feature`, senza toccare il motore. Tutte rispettano l'**anti-leakage** (solo dati passati/correnti; i livelli di struttura sono resi causali con uno shift).

In [ ]:
import sys
from pathlib import Path
for c in [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/trading-ai')]:
    if (c / 'trading_ai').exists():
        sys.path.insert(0, str(c)); break

from trading_ai.data_engine import DataEngine, generate_ohlcv
from trading_ai.feature_engineering import FeatureEngine, list_features

print('Feature disponibili:', list_features())

## 1. Dati di partenza (output Modulo 1)

Pulisci e porta i dati su un timeframe operativo (qui H1).

In [ ]:
eng = DataEngine()
raw = generate_ohlcv(n=120_000, start='2022-01-01', freq='1min', seed=11)
df = eng.load_dataframe(raw)
h1 = eng.to_timeframe(df, 'H1')
print('Candele H1:', len(h1))
h1.head()

## 2. Estrazione di TUTTE le feature

Un'unica chiamata calcola indicatori + candlestick + market structure. `dropna=True` rimuove il warm-up iniziale degli indicatori.

In [ ]:
fe = FeatureEngine()
feats = fe.transform(h1, dropna=True)
print('Shape feature:', feats.shape)
print('Colonne:', list(feats.columns))
feats.head()

## 3. Estrazione per gruppo

Puoi calcolare solo una famiglia: `indicator`, `candlestick`, `structure`, `volatility`.

In [ ]:
only_struct = fe.transform(h1, groups=['structure'], dropna=True)
print(only_struct.columns.tolist())
only_struct[['structure_bos', 'structure_choch', 'structure_trend', 'fvg_fvg']].tail(10)

## 4. Aggiungere una NUOVA feature (estensibilità)

Basta il decoratore `@feature`: da quel momento è disponibile nel motore.

In [ ]:
from trading_ai.feature_engineering import feature

@feature('hl_range_pct', group='custom')
def hl_range_pct(d):
    # Ampiezza della candela in percentuale della close: una feature di volatilità.
    return ((d['high'] - d['low']) / d['close'])

out = fe.transform(h1, features=['hl_range_pct', 'rsi'], dropna=True)
out[['hl_range_pct', 'rsi']].head()

## 5. Salvataggio matrice feature

Questa matrice è l'input del **Modulo 3 — Pattern Discovery**.

In [ ]:
import os
out_dir = Path(os.environ.get('TRADING_AI_ROOT', '.')) / 'datasets'
out_dir.mkdir(parents=True, exist_ok=True)
path = eng.save(feats, out_dir / 'SYNTH_H1_features.parquet')
print('Salvato:', path)

## ✅ Modulo 2 verificato

Indicatori, candlestick e market structure estratti correttamente, anti-leakage garantito, registry estensibile. Test: `pytest tests/test_feature_engineering.py` (10/10 verdi).

**Prossimo:** Modulo 3 — Pattern Discovery (clustering/ML non supervisionato sulle feature, con metriche di stabilità e validazione out-of-sample).